# Experiment 002 — Arm C: Volume + GMX Liquidity Filter (min $500k)

**Strategy:** IchiV2_LS_Static  
**Starting Capital:** $100,000  
**Timerange:** 2021-01-06 to 2026-03-12  
**Pairlist:** VolumePairList (top 75) + GMX liquidity filter requiring minimum $500k available liquidity.

In [ ]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import nest_asyncio

nest_asyncio.apply()
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
ARM_LABEL = 'Arm C: Volume + Liquidity'
ARM_COLOR = '#FFA15A'
TEMPLATE = 'plotly_dark'

ZIP_PATH = PROJECT_ROOT / 'experiments/ichiv2-gmx/002-volume-pairlist-layers/results/arm_c_volume_liq.zip'

print(f'Project root: {PROJECT_ROOT}')
print(f'ZIP path: {ZIP_PATH}')
print(f'ZIP exists: {ZIP_PATH.exists()}')

In [ ]:
def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades_list = data['strategy'][strategy_name]['trades']
    trades = pd.DataFrame(trades_list)
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades, strategy_name

trades, strategy_name = load_trades_from_zip(ZIP_PATH)
print(f'Strategy: {strategy_name}')
print(f'Total trades: {len(trades)}')
print(f'Date range: {trades["open_date"].min()} to {trades["close_date"].max()}')
trades.head()

In [ ]:
# --- Summary Metrics ---

def calculate_metrics(trades, starting_balance=STARTING_BALANCE):
    trades_sorted = trades.sort_values('close_date').copy()
    trades_sorted['cum_profit_abs'] = trades_sorted['profit_abs'].cumsum()
    trades_sorted['equity'] = starting_balance + trades_sorted['cum_profit_abs']
    
    total_profit_abs = trades_sorted['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100
    
    days = (trades_sorted['close_date'].max() - trades_sorted['open_date'].min()).days
    years = days / 365.25
    final_equity = starting_balance + total_profit_abs
    cagr = ((final_equity / starting_balance) ** (1 / years) - 1) * 100 if years > 0 else 0
    
    equity = trades_sorted['equity']
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max * 100
    max_dd = drawdown.min()
    
    trades_sorted['close_day'] = trades_sorted['close_date'].dt.date
    daily_pnl = trades_sorted.groupby('close_day')['profit_abs'].sum()
    daily_returns = daily_pnl / starting_balance
    
    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365) if daily_returns.std() > 0 else 0
    downside = daily_returns[daily_returns < 0]
    sortino = (daily_returns.mean() / downside.std()) * np.sqrt(365) if len(downside) > 0 and downside.std() > 0 else 0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0
    
    win_rate = (trades_sorted['profit_abs'] > 0).mean() * 100
    
    gross_profit = trades_sorted.loc[trades_sorted['profit_abs'] > 0, 'profit_abs'].sum()
    gross_loss = abs(trades_sorted.loc[trades_sorted['profit_abs'] < 0, 'profit_abs'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    avg_duration = trades_sorted['trade_duration'].mean() if 'trade_duration' in trades_sorted.columns else 0
    
    return {
        'Trades': len(trades_sorted),
        'Total Profit ($)': f'${total_profit_abs:,.2f}',
        'Total Profit (%)': f'{total_profit_pct:.2f}%',
        'CAGR (%)': f'{cagr:.2f}%',
        'Max Drawdown (%)': f'{max_dd:.2f}%',
        'Sharpe Ratio': f'{sharpe:.2f}',
        'Sortino Ratio': f'{sortino:.2f}',
        'Calmar Ratio': f'{calmar:.2f}',
        'Win Rate (%)': f'{win_rate:.2f}%',
        'Profit Factor': f'{profit_factor:.2f}',
        'Avg Trade Duration (min)': f'{avg_duration:.0f}',
    }

metrics = calculate_metrics(trades)
metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=[ARM_LABEL])
metrics_df.style.set_properties(**{'text-align': 'right'})

In [ ]:
# --- Equity Curve ---

trades_sorted = trades.sort_values('close_date').copy()
trades_sorted['cum_profit_abs'] = trades_sorted['profit_abs'].cumsum()
trades_sorted['equity'] = STARTING_BALANCE + trades_sorted['cum_profit_abs']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trades_sorted['close_date'],
    y=trades_sorted['equity'],
    mode='lines',
    name=ARM_LABEL,
    line=dict(color=ARM_COLOR, width=2),
))
fig.update_layout(
    title=f'{ARM_LABEL} — Equity Curve',
    xaxis_title='Date',
    yaxis_title='Equity ($)',
    template=TEMPLATE,
    height=500,
)
fig.show()

In [ ]:
# --- Drawdown Chart ---

rolling_max = trades_sorted['equity'].cummax()
trades_sorted['drawdown_pct'] = (trades_sorted['equity'] - rolling_max) / rolling_max * 100

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trades_sorted['close_date'],
    y=trades_sorted['drawdown_pct'],
    fill='tozeroy',
    mode='lines',
    name='Drawdown',
    line=dict(color=ARM_COLOR, width=1),
))
fig.update_layout(
    title=f'{ARM_LABEL} — Drawdown',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template=TEMPLATE,
    height=400,
)
fig.show()

In [ ]:
# --- Monthly Returns Bar Chart ---

trades_sorted['month'] = trades_sorted['close_date'].dt.to_period('M')
monthly = trades_sorted.groupby('month')['profit_abs'].sum().reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly['color'] = monthly['profit_abs'].apply(lambda x: '#00CC96' if x >= 0 else '#EF553B')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly['month_str'],
    y=monthly['profit_abs'],
    marker_color=monthly['color'],
    name='Monthly P&L',
))
fig.update_layout(
    title=f'{ARM_LABEL} — Monthly Returns',
    xaxis_title='Month',
    yaxis_title='Profit ($)',
    template=TEMPLATE,
    height=450,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
# --- Long vs Short Profit Breakdown ---

direction_col = 'is_short' if 'is_short' in trades.columns else 'trade_direction'

if direction_col == 'is_short':
    trades['direction'] = trades['is_short'].apply(lambda x: 'Short' if x else 'Long')
else:
    trades['direction'] = trades[direction_col].apply(lambda x: 'Short' if 'short' in str(x).lower() else 'Long')

long_short = trades.groupby('direction')['profit_abs'].agg(['sum', 'count']).reset_index()
long_short.columns = ['Direction', 'Total Profit', 'Trade Count']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=long_short['Direction'],
    y=long_short['Total Profit'],
    marker_color=['#636EFA', '#EF553B'],
    text=long_short['Total Profit'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
))
fig.update_layout(
    title=f'{ARM_LABEL} — Long vs Short Profit',
    xaxis_title='Direction',
    yaxis_title='Total Profit ($)',
    template=TEMPLATE,
    height=400,
)
fig.show()

In [ ]:
# --- Top 10 / Bottom 10 Pairs by Profit ---

pair_profit = trades.groupby('pair')['profit_abs'].sum().sort_values(ascending=False)

top10 = pair_profit.head(10)
bottom10 = pair_profit.tail(10)
combined = pd.concat([top10, bottom10])
combined = combined.sort_values(ascending=True)

colors = ['#00CC96' if v >= 0 else '#EF553B' for v in combined.values]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=combined.values,
    y=combined.index,
    orientation='h',
    marker_color=colors,
    text=[f'${v:,.0f}' for v in combined.values],
    textposition='outside',
))
fig.update_layout(
    title=f'{ARM_LABEL} — Top 10 & Bottom 10 Pairs by Profit',
    xaxis_title='Profit ($)',
    template=TEMPLATE,
    height=600,
    margin=dict(l=120),
)
fig.show()

In [ ]:
# --- Trade Duration Distribution ---

if 'trade_duration' in trades.columns:
    duration_hours = trades['trade_duration'] / 60
else:
    duration_hours = (trades['close_date'] - trades['open_date']).dt.total_seconds() / 3600

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=duration_hours,
    nbinsx=50,
    marker_color=ARM_COLOR,
    name='Duration',
))
fig.update_layout(
    title=f'{ARM_LABEL} — Trade Duration Distribution',
    xaxis_title='Duration (hours)',
    yaxis_title='Count',
    template=TEMPLATE,
    height=400,
)
fig.show()